# Iteration 2: Approximate Nearest Neighbour (ANN) Search

Two-stage pipeline:
1. **Stage 1** — IVF index (512 centroids) picks candidates with `nprobe=3`
2. **Stage 2** — exact Cosine Similarity on those candidates via `SQ8Store` (dequantised to float32)

In [2]:
!pip install -q torch psutil numpy faiss-cpu

In [3]:
import sys, os, time
import psutil
import numpy as np
import torch
import torch.nn.functional as F
import faiss

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


In [4]:
N_CENTROIDS = 512
N_PROBE     = 3
TOP_K       = 5
N_SYNTH     = 500_000
DIM         = 256

## Load corpus

Run `python scripts/build_storage.py` first.

In [5]:
try:
    from storage import Float32Store
    f32_store = Float32Store()
    f32_store.load()   # reads storage_data/store_float32.npy
    corpus_f32 = f32_store.data.astype(np.float32)
    corpus_f32 /= np.linalg.norm(corpus_f32, axis=1, keepdims=True)   # L2-normalise
    N, D = corpus_f32.shape
    print(f"Float32Store: {N:,} × {D} | {f32_store.get_memory_footprint():.1f} MB")
except Exception as e:
    print(f"Float32Store not available ({e}) — using synthetic data")
    rng = np.random.default_rng(42)
    corpus_f32 = rng.standard_normal((N_SYNTH, DIM)).astype(np.float32)
    corpus_f32 /= np.linalg.norm(corpus_f32, axis=1, keepdims=True)
    N, D = corpus_f32.shape

# SQ8Store is 4× smaller than float32 — used in stage 2 to retrieve candidates cheaply
try:
    from storage import SQ8Store
    sq8_store = SQ8Store()
    sq8_store.load()   # reads storage_data/store_sq8.npz
    print(f"SQ8Store: {sq8_store.quantized_data.shape[0]:,} × {sq8_store.quantized_data.shape[1]} | {sq8_store.get_memory_footprint():.1f} MB")
except Exception as e:
    print(f"SQ8Store not available ({e}) — stage 2 will use float32 corpus")
    sq8_store = None

[Float32Store] Loading from /Users/dilfd/Documents/dls_project/local-wiki-rag/storage_data/store_float32.npy...
[Float32Store] Loaded. Shape: (500000, 256)
Float32Store: 500,000 × 256 | 488.3 MB
[SQ8Store] Loading from /Users/dilfd/Documents/dls_project/local-wiki-rag/storage_data/store_sq8.npz...
[SQ8Store] Loaded. Shape: (500000, 256)
SQ8Store: 500,000 × 256 | 122.1 MB


## Train IVF index

In [ ]:
try:
    from prod.ivf_opq_pq_index import IVFOPQPQConfig, IVFOPQPQIndex

    cfg = IVFOPQPQConfig(
        dim=D,
        nlist=N_CENTROIDS,   # number of Voronoi cells (centroids)
        M=D // 8,            # number of PQ subvectors
        nbits=8,
        metric="cosine",
    )
    person4_index = IVFOPQPQIndex(cfg)
    print(f"Training IVFOPQPQIndex ({N_CENTROIDS} centroids) on {N:,} vectors...")
    t0 = time.perf_counter()
    person4_index.train_add(corpus_f32)   # k-means clustering + add all vectors
    print(f"Done in {time.perf_counter() - t0:.1f}s")

except Exception as e:
    # if prod/ is not available, fall back to plain FAISS IVF with the same setup
    print(f"IVFOPQPQIndex not available ({e}) — fallback to faiss.IndexIVFFlat")
    quantizer = faiss.IndexFlatIP(D)
    _ivf = faiss.IndexIVFFlat(quantizer, D, N_CENTROIDS, faiss.METRIC_INNER_PRODUCT)
    t0 = time.perf_counter()
    _ivf.train(corpus_f32)
    _ivf.add(corpus_f32)
    print(f"Done in {time.perf_counter() - t0:.1f}s")

    class _FallbackIndex:
        def __init__(self, idx):
            self.idx = idx
        def search(self, queries, k, nprobe=N_PROBE):
            self.idx.nprobe = nprobe
            return self.idx.search(queries, k)

    person4_index = _FallbackIndex(_ivf)

## ANNSearch

In [ ]:
class ANNSearch:
    def __init__(self, ivf_index, sq8_store, corpus_fallback=None):
        self.ivf      = ivf_index
        self.sq8      = sq8_store
        self.fallback = corpus_fallback   # used when SQ8Store is not available

    def search(self, query: np.ndarray, top_k: int = TOP_K, n_probe: int = N_PROBE):
        q = query.astype(np.float32)
        q /= np.linalg.norm(q) + 1e-12   # L2-normalise the query

        # Stage 1: IVF compares query against 512 centroids, picks n_probe nearest clusters
        # request 2× the average cluster size to cover all candidates in visited clusters
        n_candidates = min(N, n_probe * (N // N_CENTROIDS) * 2)
        _, cand_ids = self.ivf.search(q.reshape(1, -1), k=n_candidates, nprobe=n_probe)
        cand_ids = cand_ids[0][cand_ids[0] >= 0]   # FAISS pads short results with -1

        # Stage 2: exact cosine similarity only within the selected clusters
        # SQ8Store.get_vectors() dequantises uint8 → float32 on the fly
        vecs = self.sq8.get_vectors(cand_ids) if self.sq8 else self.fallback[cand_ids]
        vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
        scores = vecs @ q

        # pick top_k from the candidate set
        k = min(top_k, len(scores))
        top = np.argpartition(-scores, k - 1)[:k]
        top = top[np.argsort(-scores[top])]
        return cand_ids[top], scores[top]


ann = ANNSearch(person4_index, sq8_store, corpus_fallback=corpus_f32)
print("ANNSearch ready")

## Demo search

In [ ]:
_ = ann.search(np.random.randn(D).astype(np.float32))  # warm-up

query_vec = np.random.randn(D).astype(np.float32)

t0 = time.perf_counter()
indices, scores = ann.search(query_vec, top_k=TOP_K)
latency_ms = (time.perf_counter() - t0) * 1000

print(f"Latency : {latency_ms:.2f} ms")
print(f"Visited : {N_PROBE}/{N_CENTROIDS} clusters")
print(f"\nTop-{TOP_K} results:")
print(f"{'Rank':>4}  {'Doc ID':>8}  {'Cosine Score':>12}")
print("-" * 30)
for rank, (idx, score) in enumerate(zip(indices, scores), 1):
    print(f"{rank:>4}  {idx:>8}  {score:>12.4f}")

## Memory comparison: SQ8 vs Float32

In [ ]:
float32_mb = N * D * 4 / 1024**2
sq8_mb     = sq8_store.get_memory_footprint() if sq8_store else N * D / 1024**2

print(f"Float32 corpus : {float32_mb:.1f} MB")
print(f"SQ8 corpus     : {sq8_mb:.1f} MB  ({float32_mb / sq8_mb:.1f}× smaller)")

proc = psutil.Process(os.getpid())
print(f"Process RAM    : {proc.memory_info().rss / 1024**2:.1f} MB")